# Reginald — Failure-Case Analysis

**Task:** review the baseline U-Net's prediction failures on the held-out
test split and identify the **common segmentation mistakes**, with
representative examples the report can use directly.

This notebook is an **evaluation-only** step. It does not train or re-run
inference — it consumes the per-sample prediction PNGs that Saikiran's
evaluation notebook has already written to
`outputs/saikiran/prediction_masks/`.

See `docs/failure_analysis.md` for the full methodology.

Outputs are written under `outputs/reginald/failure_analysis/`:

- `per_sample_metrics.csv` — one row per test sample
- `aggregate_metrics.json` — mean / median / micro scores + a cross-check
  against Saikiran's headline numbers
- `confusion_matrix.png`, `score_distributions.png`, `category_counts.png`
- `cases/sample_XXXXX.png` — 5-panel figure per scored test sample
- `grids/*.png` — one grid per failure category + an overall worst-by-IoU grid
- `summary.md` — auto-generated written summary to paste into the report

## How to run

1. Ensure preprocessing has been completed (test masks exist under
   `data/processed/test/mask/`).
2. Ensure Saikiran's evaluation has been run so that binary prediction
   PNGs exist at:
   ```text
   outputs/saikiran/prediction_masks/sample_XXXXX_pred.png
   ```
   If the checkpoint is not on disk yet, request `best_model.pth` from the
   shared Google Drive, drop it into
   `outputs/prajwal/checkpoints/best_model.pth`, then run
   `saikiran_evaluation.ipynb` once.
3. Run this notebook top-to-bottom.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import rasterio
from skimage import measure, morphology


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start


PROJECT_ROOT     = find_project_root()
METADATA_PATH    = PROJECT_ROOT / "data" / "processed" / "metadata.csv"
PREDICTIONS_DIR  = PROJECT_ROOT / "outputs" / "saikiran" / "prediction_masks"
SAIKIRAN_SUMMARY = PROJECT_ROOT / "outputs" / "saikiran" / "metrics_summary.csv"
ANALYSIS_DIR     = PROJECT_ROOT / "outputs" / "reginald" / "failure_analysis"
CASES_DIR        = ANALYSIS_DIR / "cases"
GRIDS_DIR        = ANALYSIS_DIR / "grids"

for d in (ANALYSIS_DIR, CASES_DIR, GRIDS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT    :", PROJECT_ROOT)
print("METADATA_PATH   :", METADATA_PATH.exists(), METADATA_PATH)
print("PREDICTIONS_DIR :", PREDICTIONS_DIR.exists(), PREDICTIONS_DIR)
print("ANALYSIS_DIR    :", ANALYSIS_DIR)

## 2. Configuration

All knobs for the analysis live here. The defaults match Saikiran's
evaluation pipeline (`IMG_SIZE = 256`, binary prediction masks).

In [ ]:
EVAL_SIZE = 256     

# --- prediction decoding ---
PROB_THRESHOLD     = 128   
TREAT_AS_BINARY_IF = 2     
PRED_FILENAME_FMT  = "{sample_id}_pred.png"   # Saikiran's naming

# --- failure-categorisation thresholds ---
IOU_GOOD           = 0.70
IOU_ACCEPTABLE_LOW = 0.40
RECALL_MISSED      = 0.05   
PRECISION_OVERSEG  = 0.40  
OVER_AREA_RATIO    = 3.0    
UNDER_AREA_RATIO   = 0.33  
RECALL_UNDERSEG    = 0.50  
SMALL_OBJECT_PX    = 200   
RECALL_SMALL_MISS  = 0.30
FRAGMENT_RATIO     = 3.0    
BOUNDARY_WIDTH_PX  = 3      


OVERLAY_ALPHA  = 0.45
GRID_MAX_TILES = 6         
WORST_N        = 6         

print("Config loaded. Evaluation resolution:", EVAL_SIZE)

## 3. Path helpers and image / mask loaders

Loaders mirror the conventions already established in the project
(`augmentations.ipynb` uses the same percentile stretch), and everything
is resized to `EVAL_SIZE × EVAL_SIZE` so the prediction, GT and imagery
line up pixel-for-pixel.

In [ ]:
def to_abs_path(p):
    p = Path(str(p).replace("\\", "/"))
    if p.is_absolute():
        return p
    return (PROJECT_ROOT / p).resolve()


def _percentile_stretch(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    if arr.ndim == 2:
        lo, hi = np.percentile(arr, (2, 98))
        return np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)
    out = np.zeros_like(arr, dtype=np.float32)
    for c in range(arr.shape[2]):
        lo, hi = np.percentile(arr[:, :, c], (2, 98))
        out[:, :, c] = np.clip((arr[:, :, c] - lo) / (hi - lo + 1e-6), 0, 1)
    return out


def read_tif_rgb(path_str: str, size: int = EVAL_SIZE) -> np.ndarray:
    path = to_abs_path(path_str)
    with rasterio.open(path) as src:
        if src.count >= 3:
            arr = src.read([1, 2, 3])
            arr = np.transpose(arr, (1, 2, 0))
        else:
            arr = src.read(1)
            arr = np.stack([arr] * 3, axis=-1)
    arr = _percentile_stretch(arr)
    img = (arr * 255).astype(np.uint8)
    img = np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR))
    return img


def read_gt_mask(path_str: str, size: int = EVAL_SIZE) -> np.ndarray:
    path = to_abs_path(path_str)
    mask = np.array(Image.open(path).convert("L"), dtype=np.uint8)
    mask = (mask > 0).astype(np.uint8)
    if mask.shape != (size, size):
        mask = np.array(
            Image.fromarray(mask).resize((size, size), Image.NEAREST),
            dtype=np.uint8,
        )
    return (mask > 0).astype(np.uint8)


def read_prediction(path: Path, size: int = EVAL_SIZE) -> np.ndarray:
    arr = np.array(Image.open(path).convert("L"), dtype=np.uint8)
    if arr.shape != (size, size):
        arr = np.array(
            Image.fromarray(arr).resize((size, size), Image.NEAREST),
            dtype=np.uint8,
        )
    if np.unique(arr).size <= TREAT_AS_BINARY_IF:
        return (arr > 0).astype(np.uint8)
    return (arr >= PROB_THRESHOLD).astype(np.uint8)


print("Loaders ready — everything evaluates at", EVAL_SIZE, "x", EVAL_SIZE)

## 4. Metric helpers

In [ ]:
def pixel_confusion(gt: np.ndarray, pred: np.ndarray) -> dict:
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    tp = int(np.logical_and(gt, pred).sum())
    fp = int(np.logical_and(~gt, pred).sum())
    fn = int(np.logical_and(gt, ~pred).sum())
    tn = int(np.logical_and(~gt, ~pred).sum())
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}


def _safe_div(n: float, d: float) -> float:
    return float(n) / float(d) if d > 0 else float("nan")


def metrics_from_confusion(c: dict) -> dict:
    tp, fp, fn, tn = c["tp"], c["fp"], c["fn"], c["tn"]
    iou  = _safe_div(tp, tp + fp + fn)            if (tp + fp + fn)     > 0 else float("nan")
    dice = _safe_div(2 * tp, 2 * tp + fp + fn)    if (2 * tp + fp + fn) > 0 else float("nan")
    precision = _safe_div(tp, tp + fp) if (tp + fp) > 0 else float("nan")
    recall    = _safe_div(tp, tp + fn) if (tp + fn) > 0 else float("nan")
    acc       = _safe_div(tp + tn, tp + fp + fn + tn)
    return {"iou": iou, "dice": dice, "precision": precision,
            "recall": recall, "accuracy": acc}


def boundary_band(mask: np.ndarray, width: int = BOUNDARY_WIDTH_PX) -> np.ndarray:
    if mask.sum() == 0:
        return np.zeros_like(mask, dtype=bool)
    dil = morphology.binary_dilation(mask.astype(bool), morphology.disk(width))
    ero = morphology.binary_erosion(mask.astype(bool), morphology.disk(width))
    return np.logical_and(dil, np.logical_not(ero))


def boundary_iou(gt: np.ndarray, pred: np.ndarray,
                 width: int = BOUNDARY_WIDTH_PX) -> float:
    gt_b = boundary_band(gt, width)
    pred_b = boundary_band(pred, width)
    if not gt_b.any() and not pred_b.any():
        return float("nan")
    inter = np.logical_and(gt_b, pred_b).sum()
    union = np.logical_or(gt_b, pred_b).sum()
    return _safe_div(int(inter), int(union)) if union > 0 else float("nan")


def component_stats(mask: np.ndarray) -> dict:
    if mask.sum() == 0:
        return {"num_components": 0, "min_area": 0, "max_area": 0, "mean_area": 0.0}
    labelled = measure.label(mask, connectivity=2)
    regions = measure.regionprops(labelled)
    areas = [r.area for r in regions]
    return {
        "num_components": len(areas),
        "min_area": int(min(areas)),
        "max_area": int(max(areas)),
        "mean_area": float(np.mean(areas)),
    }


print("Metric helpers ready.")

## 5. Locate Saikiran's predictions and match test samples

Every test sample in `metadata.csv` is matched with its corresponding
prediction PNG in `outputs/saikiran/prediction_masks/`. Missing
predictions are reported and excluded from the scored set (but do not
abort the analysis).

In [ ]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"metadata.csv not found at {METADATA_PATH}. "
        "Run Rajesh's preprocessing first."
    )

meta = pd.read_csv(METADATA_PATH)
test_df = meta[meta["split"] == "test"].copy().reset_index(drop=True)
if "mask_status" in test_df.columns:
    test_df = test_df[test_df["mask_status"] == "OK"].reset_index(drop=True)

print("Test samples in metadata:", len(test_df))


def find_prediction(sample_id: str) -> Path | None:
    candidate = PREDICTIONS_DIR / PRED_FILENAME_FMT.format(sample_id=sample_id)
    return candidate if candidate.exists() else None


test_df["prediction_path"] = test_df["sample_id"].apply(
    lambda sid: (
        str(find_prediction(sid).relative_to(PROJECT_ROOT))
        if find_prediction(sid) else ""
    )
)

present_mask = test_df["prediction_path"] != ""
print("Predictions present :", int(present_mask.sum()))
print("Predictions missing :", int((~present_mask).sum()))
if (~present_mask).any():
    missing_ids = test_df.loc[~present_mask, "sample_id"].tolist()
    print("  missing sample ids (first 10):", missing_ids[:10])
    print(
        "  → run saikiran_evaluation.ipynb to populate",
        PREDICTIONS_DIR.relative_to(PROJECT_ROOT),
    )

test_df.head()

## 6. Per-sample metrics

Every test sample with a prediction gets one row. Samples without a
prediction are recorded with `status = 'missing_prediction'`.

In [ ]:
rows = []
for _, row in test_df.iterrows():
    sample_id = row["sample_id"]
    gt = read_gt_mask(row["mask_path"])

    pred_path = find_prediction(sample_id)
    if pred_path is None:
        rows.append({"sample_id": sample_id, "status": "missing_prediction"})
        continue

    pred = read_prediction(pred_path)

    conf = pixel_confusion(gt, pred)
    m    = metrics_from_confusion(conf)
    b_iou = boundary_iou(gt, pred)
    gt_stats   = component_stats(gt)
    pred_stats = component_stats(pred)

    n_pixels  = int(gt.size)
    gt_area   = int(gt.sum())
    pred_area = int(pred.sum())

    rows.append({
        "sample_id": sample_id,
        "status": "scored",
        "mask_path": row["mask_path"],
        "pre_path": row["pre_path"],
        "post_path": row["post_path"],
        "prediction_path": str(pred_path.relative_to(PROJECT_ROOT)),
        "n_pixels": n_pixels,
        "gt_area_px": gt_area,
        "pred_area_px": pred_area,
        "gt_area_frac":   gt_area   / n_pixels,
        "pred_area_frac": pred_area / n_pixels,
        "area_ratio": (pred_area / gt_area) if gt_area > 0 else float("inf"),
        **conf,
        **m,
        "boundary_iou": b_iou,
        "gt_num_components":   gt_stats["num_components"],
        "gt_min_area":         gt_stats["min_area"],
        "gt_mean_area":        gt_stats["mean_area"],
        "pred_num_components": pred_stats["num_components"],
    })

per_sample = pd.DataFrame(rows)
per_sample_path = ANALYSIS_DIR / "per_sample_metrics.csv"
per_sample.to_csv(per_sample_path, index=False)

print("Scored samples   :", (per_sample["status"] == "scored").sum())
print("Missing samples  :", (per_sample["status"] == "missing_prediction").sum())
print("Saved            :", per_sample_path)
per_sample.head()

## 7. Aggregate metrics, confusion matrix, and cross-check with Saikiran

We recompute the headline numbers from the prediction PNGs and verify
that they line up with Saikiran's `metrics_summary.csv`. If they are
close, the analysis is trustworthy; if they are off, something went wrong
in the resolution handling.

In [ ]:
scored = per_sample[per_sample["status"] == "scored"].copy()

if scored.empty:
    print("No scored samples — drop predictions into",
          PREDICTIONS_DIR.relative_to(PROJECT_ROOT), "and re-run.")
else:
    micro = {
        "tp": int(scored["tp"].sum()),
        "fp": int(scored["fp"].sum()),
        "fn": int(scored["fn"].sum()),
        "tn": int(scored["tn"].sum()),
    }
    micro_metrics = metrics_from_confusion(micro)

    aggregate = {
        "n_scored":  int(len(scored)),
        "n_missing": int((per_sample["status"] == "missing_prediction").sum()),
        "mean_iou":       float(scored["iou"].mean(skipna=True)),
        "median_iou":     float(scored["iou"].median(skipna=True)),
        "mean_dice":      float(scored["dice"].mean(skipna=True)),
        "mean_precision": float(scored["precision"].mean(skipna=True)),
        "mean_recall":    float(scored["recall"].mean(skipna=True)),
        "mean_accuracy":  float(scored["accuracy"].mean(skipna=True)),
        "mean_boundary_iou": float(scored["boundary_iou"].mean(skipna=True)),
        "micro": {**micro, **micro_metrics},
    }

    # Cross-check against Saikiran's reported numbers
    if SAIKIRAN_SUMMARY.exists():
        sk = pd.read_csv(SAIKIRAN_SUMMARY)
        if "Test" in sk.columns:
            sk_lookup = dict(zip(sk["Metric"], sk["Test"]))
            deltas = {}
            for k, col in [("iou", "IoU"), ("dice", "Dice/F1"),
                           ("precision", "Precision"), ("recall", "Recall")]:
                if col in sk_lookup:
                    deltas[col] = {
                        "saikiran": float(sk_lookup[col]),
                        "here_micro": float(micro_metrics[k]),
                        "delta":     float(micro_metrics[k] - float(sk_lookup[col])),
                    }
            aggregate["saikiran_cross_check"] = deltas
            print("Cross-check vs Saikiran (Test split):")
            for k, v in deltas.items():
                print(f"  {k:10s}: saikiran={v['saikiran']:.4f}  "
                      f"here={v['here_micro']:.4f}  delta={v['delta']:+.4f}")
    else:
        print("Note: Saikiran's metrics_summary.csv not found, cross-check skipped.")

    with open(ANALYSIS_DIR / "aggregate_metrics.json", "w") as f:
        json.dump(aggregate, f, indent=2)

    print()
    print(json.dumps({k: v for k, v in aggregate.items()
                      if k != "saikiran_cross_check"}, indent=2))

    cm = np.array([[micro["tn"], micro["fp"]],
                   [micro["fn"], micro["tp"]]])
    fig, ax = plt.subplots(figsize=(4.5, 4.0))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticklabels(["GT 0",   "GT 1"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    color="black" if cm[i, j] < cm.max() / 2 else "white")
    ax.set_title("Pixel-level confusion matrix (test, micro)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(ANALYSIS_DIR / "confusion_matrix.png", dpi=160)
    plt.show()

## 8. Score distributions

Histogram of per-sample IoU / Dice / precision / recall. The **left tail**
of the IoU histogram is where the failure cases live.

In [ ]:
if not scored.empty:
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    for ax, (col, title) in zip(axes.flat, [
        ("iou", "IoU"), ("dice", "Dice"),
        ("precision", "Precision"), ("recall", "Recall"),
    ]):
        vals = scored[col].dropna().values
        ax.hist(vals, bins=20, range=(0, 1), color="#3b7dd8", edgecolor="white")
        mean_val = float(np.nanmean(vals)) if len(vals) else float("nan")
        ax.axvline(mean_val, color="red", linestyle="--", linewidth=1,
                   label=f"mean = {mean_val:.3f}")
        ax.set_title(title)
        ax.set_xlim(0, 1)
        ax.set_xlabel(title)
        ax.set_ylabel("samples")
        ax.legend()
    fig.suptitle("Per-sample score distribution on the test split", y=1.02)
    fig.tight_layout()
    fig.savefig(ANALYSIS_DIR / "score_distributions.png", dpi=160, bbox_inches="tight")
    plt.show()

## 9. Failure categorisation

Each scored sample is assigned a single primary label. Rules are evaluated
in the order defined in section 2 — first match wins. The exact thresholds
used are printed with each sample in the per-sample CSV.

In [ ]:
def categorise(row: pd.Series) -> str:
    gt_area   = row["gt_area_px"]
    pred_area = row["pred_area_px"]
    iou       = row["iou"]
    precision = row["precision"]
    recall    = row["recall"]
    b_iou     = row["boundary_iou"]
    gt_ncomp   = row["gt_num_components"]
    pred_ncomp = row["pred_num_components"]

    if gt_area == 0 and pred_area == 0:
        return "correct_empty"
    if gt_area == 0 and pred_area > 0:
        return "no_gt_with_fp"

    if not np.isnan(recall) and recall < RECALL_MISSED:
        return "missed_detection"

    area_ratio = pred_area / gt_area if gt_area > 0 else float("inf")

    if (area_ratio >= OVER_AREA_RATIO) and (not np.isnan(precision)) and (precision < PRECISION_OVERSEG):
        return "over_segmentation"
    if (area_ratio <= UNDER_AREA_RATIO) and (not np.isnan(recall)) and (recall < RECALL_UNDERSEG):
        return "under_segmentation"

    if row["gt_mean_area"] > 0 and row["gt_mean_area"] < SMALL_OBJECT_PX        and (not np.isnan(recall)) and recall < RECALL_SMALL_MISS:
        return "small_object_miss"

    if gt_ncomp > 0 and pred_ncomp > FRAGMENT_RATIO * max(gt_ncomp, 1)        and (not np.isnan(iou)) and iou < 0.5:
        return "fragmented_prediction"

    if (not np.isnan(iou)) and IOU_ACCEPTABLE_LOW <= iou < IOU_GOOD        and (not np.isnan(b_iou)) and b_iou < iou - 0.15:
        return "boundary_error"

    if not np.isnan(iou):
        if iou >= IOU_GOOD:
            return "good"
        if iou >= IOU_ACCEPTABLE_LOW:
            return "acceptable"
        return "poor"
    return "poor"


if not scored.empty:
    scored = scored.copy()
    scored["category"] = scored.apply(categorise, axis=1)
    per_sample.loc[scored.index, "category"] = scored["category"]
    per_sample.to_csv(per_sample_path, index=False)

    cat_counts = scored["category"].value_counts()
    print("Category counts:")
    print(cat_counts.to_string())

    fig, ax = plt.subplots(figsize=(8, 4.5))
    cat_counts.sort_values().plot.barh(ax=ax, color="#d8743b")
    ax.set_xlabel("number of test samples")
    ax.set_title("Failure / quality categories on the test split")
    for i, v in enumerate(cat_counts.sort_values().values):
        ax.text(v + 0.1, i, str(int(v)), va="center")
    fig.tight_layout()
    fig.savefig(ANALYSIS_DIR / "category_counts.png", dpi=160)
    plt.show()

## 10. Per-sample qualitative figures

For every scored sample a single 5-panel PNG is saved to
`outputs/reginald/failure_analysis/cases/`:

1. pre-event RGB
2. post-event RGB
3. ground-truth overlaid on post
4. prediction overlaid on post
5. error map — `TP = green`, `FP = red`, `FN = blue`

In [ ]:
TP_COLOR = np.array([ 46, 204,  64], dtype=np.float32)   # green
FP_COLOR = np.array([230,  60,  60], dtype=np.float32)   # red
FN_COLOR = np.array([ 60, 120, 230], dtype=np.float32)   # blue


def overlay_mask(image: np.ndarray, mask: np.ndarray,
                 color=(46, 204, 64), alpha=OVERLAY_ALPHA) -> np.ndarray:
    out = image.astype(np.float32).copy()
    color = np.array(color, dtype=np.float32)
    sel = mask.astype(bool)
    out[sel] = (1 - alpha) * out[sel] + alpha * color
    return np.clip(out, 0, 255).astype(np.uint8)


def error_map(image: np.ndarray, gt: np.ndarray, pred: np.ndarray,
              alpha: float = 0.55) -> np.ndarray:
    out = image.astype(np.float32).copy()
    gt_b   = gt.astype(bool)
    pred_b = pred.astype(bool)
    tp = np.logical_and(gt_b, pred_b)
    fp = np.logical_and(~gt_b, pred_b)
    fn = np.logical_and(gt_b, ~pred_b)
    for sel, color in ((tp, TP_COLOR), (fp, FP_COLOR), (fn, FN_COLOR)):
        if sel.any():
            out[sel] = (1 - alpha) * out[sel] + alpha * color
    return np.clip(out, 0, 255).astype(np.uint8)


def render_case_figure(row: pd.Series, out_path: Path):
    pre  = read_tif_rgb(row["pre_path"])
    post = read_tif_rgb(row["post_path"])
    gt   = read_gt_mask(row["mask_path"])
    pred = read_prediction(to_abs_path(row["prediction_path"]))

    gt_over   = overlay_mask(post, gt,   color=(60, 180, 255))
    pred_over = overlay_mask(post, pred, color=(255, 190, 40))
    err       = error_map(post, gt, pred)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
    titles = ["pre-event", "post-event", "ground truth", "prediction", "error map"]
    imgs   = [pre, post, gt_over, pred_over, err]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    legend_patches = [
        mpatches.Patch(color=TP_COLOR / 255, label="TP"),
        mpatches.Patch(color=FP_COLOR / 255, label="FP"),
        mpatches.Patch(color=FN_COLOR / 255, label="FN"),
    ]
    axes[-1].legend(handles=legend_patches, loc="lower right",
                    fontsize=8, framealpha=0.8)

    iou = row["iou"]; dice = row["dice"]
    prec = row["precision"]; rec = row["recall"]
    cat = row.get("category", "")
    fig.suptitle(
        f"{row['sample_id']}  |  IoU={iou:.3f}  Dice={dice:.3f}  "
        f"P={prec:.3f}  R={rec:.3f}  |  {cat}",
        y=1.04, fontsize=11,
    )
    fig.tight_layout()
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


if not scored.empty:
    rendered = 0
    for _, row in scored.iterrows():
        out_path = CASES_DIR / f"{row['sample_id']}.png"
        try:
            render_case_figure(row, out_path)
            rendered += 1
        except Exception as e:
            print(f"Failed to render {row['sample_id']}: {e}")
    print(f"Rendered {rendered} per-sample figures to {CASES_DIR}")

## 11. Worst-case ranking and category grids

- `grids/worst_by_iou.png` — the overall N worst samples on the test split
- `grids/category_<name>.png` — up to 6 representative samples per failure
  category (sorted by IoU ascending, so the most instructive cases come first)

In [ ]:
def make_grid_figure(sub_df: pd.DataFrame, title: str, out_path: Path,
                     max_tiles: int = GRID_MAX_TILES):
    sub_df = sub_df.head(max_tiles)
    n = len(sub_df)
    if n == 0:
        return
    cols = min(3, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 5.0 * rows),
                             squeeze=False)

    for ax_row in axes:
        for ax in ax_row:
            ax.axis("off")

    for idx, (_, row) in enumerate(sub_df.iterrows()):
        r, c = divmod(idx, cols)
        ax = axes[r][c]
        try:
            post = read_tif_rgb(row["post_path"])
            gt   = read_gt_mask(row["mask_path"])
            pred = read_prediction(to_abs_path(row["prediction_path"]))
            ax.imshow(error_map(post, gt, pred))
            cat = row.get("category", "")
            ax.set_title(
                f"{row['sample_id']}  IoU={row['iou']:.2f}\n{cat}",
                fontsize=9,
            )
        except Exception:
            ax.set_title(f"{row['sample_id']} (render failed)", fontsize=9)

    legend_patches = [
        mpatches.Patch(color=TP_COLOR / 255, label="TP"),
        mpatches.Patch(color=FP_COLOR / 255, label="FP"),
        mpatches.Patch(color=FN_COLOR / 255, label="FN"),
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.02), frameon=False)
    fig.suptitle(title, y=1.02, fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


if not scored.empty:
    worst = scored.sort_values("iou", ascending=True).head(WORST_N)
    make_grid_figure(
        worst,
        f"Worst {len(worst)} test samples by IoU",
        GRIDS_DIR / "worst_by_iou.png",
    )
    print("Worst-by-IoU grid saved.")

    failure_cats = [c for c in scored["category"].unique()
                    if c not in ("good", "correct_empty")]
    for cat in failure_cats:
        sub = scored[scored["category"] == cat].sort_values("iou", ascending=True)
        make_grid_figure(
            sub,
            f"Category: {cat}  ({len(sub)} samples)",
            GRIDS_DIR / f"category_{cat}.png",
        )
    print(f"Saved {len(failure_cats)} category grids to {GRIDS_DIR}")

## 12. Auto-generated written summary

Writes `outputs/reginald/failure_analysis/summary.md`, which can be pasted
directly into the group report as the *failure-case analysis* section and
then tightened up.

In [ ]:
CATEGORY_DESCRIPTIONS = {
    "correct_empty":         "empty ground truth correctly predicted empty (true negative scene)",
    "good":                  "IoU >= {:.2f} — acceptable prediction".format(IOU_GOOD),
    "acceptable":            "IoU in [{:.2f}, {:.2f}) — mediocre prediction".format(IOU_ACCEPTABLE_LOW, IOU_GOOD),
    "poor":                  "IoU < {:.2f} with no specific failure pattern".format(IOU_ACCEPTABLE_LOW),
    "no_gt_with_fp":         "model hallucinates flood damage on a clean (empty-GT) scene",
    "missed_detection":      "model predicts (almost) nothing on a scene that has flood damage",
    "over_segmentation":     "prediction area much larger than GT — precision collapses",
    "under_segmentation":    "prediction area much smaller than GT — recall collapses",
    "small_object_miss":     "GT is composed of small objects that the model fails to recover",
    "fragmented_prediction": "prediction is broken into many more components than GT",
    "boundary_error":        "mask overlaps GT but boundary is badly localised",
}

CATEGORY_FIXES = {
    "no_gt_with_fp":         "reduce false-positive rate — more clean (empty-GT) training scenes, raise decision threshold, or make the PRE-vs-POST change cue more explicit (difference channel / Siamese encoder).",
    "missed_detection":      "address class imbalance — focal / Tversky loss, oversample flooded scenes, or lower the decision threshold.",
    "over_segmentation":     "raise decision threshold, stronger regularisation, larger context window, or add a water-index (NDWI) channel.",
    "under_segmentation":    "lower the decision threshold, use multi-scale features, or train with a recall-weighted loss.",
    "small_object_miss":     "train at a higher input resolution (>256), add small-object-preserving augmentations, or use a feature-pyramid decoder.",
    "fragmented_prediction": "post-process with morphological opening / minimum-component-size filter, or add a boundary-aware / connectivity loss.",
    "boundary_error":        "add a boundary loss (e.g. BoundaryIoU or Active-Contour), train on tighter crops, or raise input resolution.",
    "poor":                  "inspect qualitatively — likely a mix of the above.",
    "acceptable":            "usually fine; incremental gains from better augmentation.",
    "good":                  "no action needed on these samples.",
    "correct_empty":         "no action needed.",
}


def write_summary():
    if scored.empty:
        (ANALYSIS_DIR / "summary.md").write_text(
            "# Failure-case analysis\n\nNo scored samples — populate "
            f"`{PREDICTIONS_DIR.relative_to(PROJECT_ROOT)}` and re-run.\n"
        )
        print("Empty summary written.")
        return

    agg = json.loads((ANALYSIS_DIR / "aggregate_metrics.json").read_text())
    cat_counts = scored["category"].value_counts()

    lines: list[str] = []
    lines.append("# Failure-case analysis — auto-generated summary\n")
    lines.append("_Model: U-Net baseline (Prajwal), evaluated by Saikiran, failure-case analysis by Reginald._\n")
    lines.append("## Headline numbers (test split)\n")
    lines.append(f"- scored test samples : **{agg['n_scored']}**")
    lines.append(f"- missing predictions : {agg['n_missing']}")
    lines.append(f"- mean IoU            : **{agg['mean_iou']:.3f}** "
                 f"(median {agg['median_iou']:.3f})")
    lines.append(f"- mean Dice           : **{agg['mean_dice']:.3f}**")
    lines.append(f"- mean precision      : {agg['mean_precision']:.3f}")
    lines.append(f"- mean recall         : {agg['mean_recall']:.3f}")
    lines.append(f"- mean boundary-IoU   : {agg['mean_boundary_iou']:.3f}")
    lines.append("")

    if "saikiran_cross_check" in agg:
        lines.append("### Cross-check vs Saikiran's metrics_summary.csv\n")
        lines.append("| metric | Saikiran (Test) | this notebook (micro) | delta |")
        lines.append("| --- | ---: | ---: | ---: |")
        for k, v in agg["saikiran_cross_check"].items():
            lines.append(
                f"| {k} | {v['saikiran']:.4f} | {v['here_micro']:.4f} | {v['delta']:+.4f} |"
            )
        lines.append("")

    lines.append("![confusion matrix](confusion_matrix.png)\n")
    lines.append("![score distributions](score_distributions.png)\n")

    lines.append("## Failure-mode breakdown\n")
    lines.append("| category | count | share | meaning |")
    lines.append("| --- | ---: | ---: | --- |")
    total = int(cat_counts.sum())
    for cat, count in cat_counts.items():
        share = count / total
        desc = CATEGORY_DESCRIPTIONS.get(cat, "")
        lines.append(f"| `{cat}` | {count} | {share:.0%} | {desc} |")
    lines.append("")
    lines.append("![category counts](category_counts.png)\n")

    lines.append("## Worst samples by IoU\n")
    lines.append("![worst by IoU](grids/worst_by_iou.png)\n")
    worst = scored.sort_values("iou", ascending=True).head(WORST_N)
    lines.append("| sample | IoU | Dice | P | R | category |")
    lines.append("| --- | ---: | ---: | ---: | ---: | --- |")
    for _, row in worst.iterrows():
        lines.append(
            f"| `{row['sample_id']}` | {row['iou']:.3f} | {row['dice']:.3f} | "
            f"{row['precision']:.3f} | {row['recall']:.3f} | `{row['category']}` |"
        )
    lines.append("")

    lines.append("## Category examples\n")
    ordered = [c for c in cat_counts.index if c not in ("good", "correct_empty")]
    for cat in ordered:
        grid_file = GRIDS_DIR / f"category_{cat}.png"
        if not grid_file.exists():
            continue
        n = int(cat_counts[cat])
        lines.append(f"### `{cat}` — {n} samples\n")
        lines.append(CATEGORY_DESCRIPTIONS.get(cat, ""))
        lines.append("")
        lines.append(f"![{cat}](grids/{grid_file.name})\n")
        lines.append(f"**Suggested fix.** {CATEGORY_FIXES.get(cat, '')}\n")

    lines.append("## Recommended next experiments for Divya's improved model\n")
    top_failures = [c for c in cat_counts.index
                    if c not in ("good", "correct_empty", "acceptable")][:3]
    if top_failures:
        lines.append(
            "The three most frequent non-good categories suggest the following priorities:\n"
        )
        for i, cat in enumerate(top_failures, 1):
            lines.append(f"{i}. **{cat}** — {CATEGORY_FIXES.get(cat, '')}")
    else:
        lines.append("No dominant failure mode detected.")
    lines.append("")

    (ANALYSIS_DIR / "summary.md").write_text("\n".join(lines))
    print("Wrote", ANALYSIS_DIR / "summary.md")


write_summary()

---

## Done

When this notebook has finished:

- `outputs/reginald/failure_analysis/summary.md` is ready to paste into the
  group report (tighten the prose, then attach the images it references).
- `outputs/reginald/failure_analysis/cases/` contains one 5-panel figure per
  test sample for deep inspection.
- `outputs/reginald/failure_analysis/grids/` contains one grid per failure
  category, plus the overall worst-by-IoU grid.

If the numbers disagree materially with Saikiran's `metrics_summary.csv`,
revisit the *Configuration* cell — in particular `EVAL_SIZE` (should match
`IMG_SIZE` in Saikiran's notebook, 256 by default) and `PROB_THRESHOLD`.